# Create Analysis Plots

Create horizontal barplots comparing model performance across metrics for each test dataset

In [ ]:
# Parameters from Snakemake
benchmarks_dir = snakemake.params.benchmarks_dir
model_mappings = snakemake.params.model_mappings  # [snakemake.wildcards.test_dataset]

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import json
from pathlib import Path
from collections import defaultdict

In [ ]:
df = {}
for i, model_type in enumerate(model_mappings.keys()):
    data = {}
    for benchmark, fns in snakemake.input.items():
        with open(fns[i]) as f:
            if benchmark == "hest_results":
                v = {"hest": json.load(f)["overall_performance"]}
            if benchmark == "musk_results":
                v = json.load(f)["task_summaries"]["zeroshot_classification"][
                    "balanced_acc"
                ]
            if benchmark == "lung_results":
                tmpdf = pd.read_csv(f)
                v = {}

                for _, row in tmpdf.iterrows():
                    prefix = f"{row['dataset']}_{row['metadata_col']}"
                    v[f"{prefix}_accuracy"] = row["accuracy"]
                    v[f"{prefix}_f1_weighted"] = row["f1_weighted"]
                    v[f"{prefix}_f1_macro"] = row["f1_macro"]

            if benchmark == "retrieval_results":
                v = (
                    pd.read_csv(f, index_col=0)
                    .loc[snakemake.wildcards.test_dataset]
                    .to_dict()
                )
            if benchmark == "cwevals_results":
                v = pd.read_csv(f, index_col=0).iloc[:, 0].to_dict()
        data.update(v)

    df[model_type] = data

df = pd.DataFrame(df)
df

In [ ]:
# Remove metrics that are all NaN
# df = df.dropna(how="all")
df = df.loc[snakemake.params.metrics]

In [ ]:
# Create the plot
fig, ax = plt.subplots(figsize=(12, max(16, len(df) * 0.4)))

# Create horizontal bar plot
x_pos = np.arange(len(df))
bar_width = 0.15
colors = ["#e74c3c", "#3498db", "#2ecc71", "#f39c12"]

for i, (model, color) in enumerate(zip(df.columns, colors)):
    if model in df.columns:
        values = df[model].values
        bars = ax.barh(
            x_pos + i * bar_width,
            values,
            bar_width,
            label=model,
            color=color,
            alpha=0.8,
        )

        # Add value labels on bars
        for j, (bar, val) in enumerate(zip(bars, values)):
            if not np.isnan(val):
                ax.text(
                    bar.get_width() + 0.001,
                    bar.get_y() + bar.get_height() / 2,
                    f"{val:.3f}",
                    ha="left",
                    va="center",
                    fontsize=8,
                )

# Customize plot
ax.set_yticks(x_pos + bar_width * 1.5)
ax.set_yticklabels(
    [metric.replace("_", " ").title() for metric in df.index], fontsize=10
)
ax.set_xlabel("Performance Score", fontsize=12)
ax.set_title(
    f'Model Performance Comparison - {snakemake.wildcards.test_dataset.replace("__", " & ").replace("_", " ").title()}',
    fontsize=14,
    fontweight="bold",
)

# Add legend with reversed order
handles, labels = ax.get_legend_handles_labels()
ax.legend(
    handles[::-1], labels[::-1], bbox_to_anchor=(1.05, 1), loc="upper left", fontsize=10
)

# Grid for better readability
ax.grid(axis="x", alpha=0.3)
ax.set_axisbelow(True)

# Adjust layout and save
plt.tight_layout()
plt.savefig(snakemake.output.plots, dpi=300, bbox_inches="tight")
# plt.close()